In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

N_CUSTOMERS = 1000
N_ORDERS    = 8000
START_DATE  = datetime(2023, 1, 1)
END_DATE    = datetime(2024, 12, 31)

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

regions = ['Tel Aviv', 'Center', 'North', 'South', 'Jerusalem']
age_groups = ['18-25', '26-35', '36-50', '50+']
genders = ['M', 'F']
tiers = ['Bronze', 'Silver', 'Gold', 'Platinum']
tier_weights = [0.50, 0.30, 0.15, 0.05]

cities_by_region = {
    'Tel Aviv':  ['Tel Aviv', 'Ramat Gan', 'Givatayim', 'Bat Yam'],
    'Center':    ['Petah Tikva', 'Rishon LeZion', 'Rehovot', 'Holon'],
    'North':     ['Haifa', 'Nazareth', 'Acre', 'Tiberias'],
    'South':     ['Beer Sheva', 'Eilat', 'Ashkelon', 'Ashdod'],
    'Jerusalem': ['Jerusalem', 'Beit Shemesh', 'Maale Adumim']
}

customers = []
for cid in range(1, N_CUSTOMERS + 1):
    reg  = random.choice(regions)
    city = random.choice(cities_by_region[reg])
    reg_date = random_date(START_DATE - timedelta(days=365), START_DATE)
    customers.append({
        'customer_id':       cid,
        'registration_date': reg_date.strftime('%Y-%m-%d'),
        'city':              city,
        'region':            reg,
        'age_group':         random.choice(age_groups),
        'gender':            random.choice(genders),
        'loyalty_tier':      np.random.choice(tiers, p=tier_weights)
    })

df_customers = pd.DataFrame(customers)
customer_lookup = dict(zip(df_customers['customer_id'], df_customers['region']))

categories = {
    'Electronics': {
        'products':    ['Laptop Stand', 'Wireless Mouse', 'USB Hub',
                        'Bluetooth Speaker', 'Phone Charger', 'Webcam'],
        'price_range': (50, 800),
        'weight':      0.25
    },
    'Home': {
        'products':    ['Towel Set', 'Bed Sheets', 'Kitchen Organizer',
                        'Candle Set', 'Storage Box', 'Throw Pillow'],
        'price_range': (30, 300),
        'weight':      0.35
    },
    'Beauty': {
        'products':    ['Face Serum', 'Shampoo', 'Moisturizer',
                        'Perfume', 'Lip Balm', 'Eye Cream'],
        'price_range': (20, 250),
        'weight':      0.25
    },
    'Sport': {
        'products':    ['Yoga Mat', 'Resistance Bands', 'Water Bottle',
                        'Gym Gloves', 'Jump Rope', 'Foam Roller'],
        'price_range': (25, 200),
        'weight':      0.15
    }
}

channels        = ['Web', 'App', 'Phone']
channel_weights = [0.55, 0.38, 0.07]

month_weights = [6, 5, 6, 5, 6, 5, 5, 6, 7, 8, 14, 12]
total         = sum(month_weights)
month_probs   = [w / total for w in month_weights]

months    = np.random.choice(range(1, 13), size=N_ORDERS, p=month_probs)
years     = np.random.choice([2023, 2024], size=N_ORDERS)
days      = np.random.randint(1, 29, size=N_ORDERS)
cids      = np.random.randint(1, N_CUSTOMERS + 1, size=N_ORDERS)
chans     = np.random.choice(channels, size=N_ORDERS, p=channel_weights)

cat_names   = list(categories.keys())
cat_probs   = [v['weight'] for v in categories.values()]
chosen_cats = np.random.choice(cat_names, size=N_ORDERS, p=cat_probs)

discount_options = [0, 0.05, 0.10, 0.15, 0.20, 0.30]
disc_weights     = [40, 20, 18, 12, 7, 3]
disc_total       = sum(disc_weights)
disc_probs       = [w / disc_total for w in disc_weights]
discounts        = np.random.choice(discount_options, size=N_ORDERS, p=disc_probs)

qty_options = [1, 2, 3, 4, 5]
qty_weights = [50, 25, 12, 8, 5]
qty_total   = sum(qty_weights)
qty_probs   = [w / qty_total for w in qty_weights]
quantities  = np.random.choice(qty_options, size=N_ORDERS, p=qty_probs)

orders = []
for i in range(N_ORDERS):
    try:
        order_dt = datetime(int(years[i]), int(months[i]), int(days[i]))
    except ValueError:
        order_dt = datetime(int(years[i]), int(months[i]), 28)

    if order_dt < START_DATE or order_dt > END_DATE:
        order_dt = random_date(START_DATE, END_DATE)

    cat_name = chosen_cats[i]
    cat      = categories[cat_name]
    product  = random.choice(cat['products'])
    price    = round(random.uniform(*cat['price_range']), 2)
    qty      = int(quantities[i])
    discount = float(discounts[i])
    revenue  = round(price * qty * (1 - discount), 2)
    cid      = int(cids[i])

    orders.append({
        'order_id':     i + 1,
        'customer_id':  cid,
        'order_date':   order_dt.strftime('%Y-%m-%d'),
        'category':     cat_name,
        'product_name': product,
        'quantity':     qty,
        'unit_price':   price,
        'discount_pct': discount,
        'revenue':      revenue,
        'region':       customer_lookup[cid],
        'channel':      chans[i]
    })

df_orders = pd.DataFrame(orders)

return_reasons = ['Damaged', 'Wrong item', 'Not satisfied', 'Changed mind', 'Other']
reason_weights = [0.30, 0.25, 0.25, 0.15, 0.05]

return_order_ids = np.random.choice(
    df_orders['order_id'].values,
    size=int(N_ORDERS * 0.08),
    replace=False
)

order_lookup = df_orders.set_index('order_id')[
    ['customer_id', 'order_date', 'revenue']
].to_dict('index')

returns = []
for rid, oid in enumerate(return_order_ids, 1):
    row       = order_lookup[oid]
    order_dt  = datetime.strptime(row['order_date'], '%Y-%m-%d')
    return_dt = order_dt + timedelta(days=random.randint(1, 30))
    if return_dt > END_DATE:
        return_dt = END_DATE

    returns.append({
        'return_id':     rid,
        'order_id':      oid,
        'customer_id':   row['customer_id'],
        'return_date':   return_dt.strftime('%Y-%m-%d'),
        'return_reason': np.random.choice(return_reasons, p=reason_weights),
        'return_amount': round(row['revenue'] * random.uniform(0.5, 1.0), 2)
    })

df_returns = pd.DataFrame(returns)

df_customers.to_csv('customers.csv', index=False)
df_orders.to_csv('orders.csv',       index=False)
df_returns.to_csv('returns.csv',     index=False)

print("✅ Dataset generated successfully!")
print(f"   customers     : {len(df_customers):,} rows")
print(f"   orders        : {len(df_orders):,} rows")
print(f"   returns       : {len(df_returns):,} rows")
print(f"   Total revenue : ₪{df_orders['revenue'].sum():,.0f}")
print(f"   Date range    : {df_orders['order_date'].min()} → {df_orders['order_date'].max()}")
import os
print("Files saved to:", os.getcwd())

✅ Dataset generated successfully!
   customers     : 1,000 rows
   orders        : 8,000 rows
   returns       : 640 rows
   Total revenue : ₪3,065,652
   Date range    : 2023-01-01 → 2024-12-28
Files saved to: c:\Users\Lera\AppData\Local\Programs\Microsoft VS Code
